# Addition matrix 1 · The matrix, and the columns as rotors
**Engine:** `ValaQuenta.modules.addition_matrix` · confidence floor ESTABLISHED (each result is computed here, exhaustively over the stated range).

Grade-school multiplication writes one row per digit of `num2`; adding the rows column by column gives the product. This notebook runs the core of the Multiplication Matrix Explorer and then reads the columns as rotors: the digit in column *c* has an exact period, `x → a·x + b` on ℤ/10ᵐ factors through ℤ/2ᵐ × ℤ/5ᵐ, and the odometer returns to a coarser odometer.

In [1]:
from ValaQuenta.modules.addition_matrix import maths as M
r = M.carry_matrix(1234567890, 9876543210)
print('product from the matrix:', r[1], '| Python:', 1234567890*9876543210, '| max carry in the whole matrix:', max(max(row) for row in r[0]))

product from the matrix: 12193263111263526900 | Python: 12193263111263526900 | max carry in the whole matrix: 1


**Column periods.** Column *c* of `num1 × num2` depends on `num2` only modulo 10^(c+1); its exact period is 10^(c+1)/gcd(num1, 10^(c+1)). The measured value tests every divisor over a full range.

In [2]:
for num1 in (1, 3, 7, 12, 25, 64, 625):
    print(num1, [(M.column_period(num1, c), M.column_period_measured(num1, c)) for c in range(3)])

1 [(10, 10), (100, 100), (1000, 1000)]
3 [(10, 10), (100, 100), (1000, 1000)]
7 [(10, 10), (100, 100), (1000, 1000)]
12 [(5, 5), (25, 25), (250, 250)]
25 [(2, 2), (4, 4), (40, 40)]
64 [(5, 5), (25, 25), (125, 125)]
625 [(2, 2), (4, 4), (8, 8)]


**Two rotor stacks (CRT).** The cycle type of `x → a·x + b` on ℤ/10ᵐ equals the combination of its types on ℤ/2ᵐ and ℤ/5ᵐ: lengths l₁, l₂ give gcd(l₁,l₂) cycles of length lcm(l₁,l₂). Checked for every unit multiplier a, and a spread of b, at m = 1, 2, 3.

In [3]:
from math import gcd
bad = 0; n = 0
for m in (1, 2, 3):
    Mm = 10**m
    for a in range(1, Mm):
        if gcd(a, 10) != 1: continue
        for b in range(0, Mm, max(1, Mm//40)):
            n += 1; bad += (not M.crt_rule_check(a, b, m))
print(n, 'maps checked; failures:', bad)
print('x -> 3x + 1 on Z/1000:', M.affine_cycle_type(3, 1, 1000))
print('  on Z/8   :', M.affine_cycle_type(3, 1, 8), ' on Z/125 :', M.affine_cycle_type(3, 1, 125))

18040 maps checked; failures: 0
x -> 3x + 1 on Z/1000: {100: 8, 20: 8, 4: 10}
  on Z/8   : {4: 2}  on Z/125 : {100: 1, 20: 1, 4: 1, 1: 1}


**Odometer.** `x → x + 1` on ℤ/10ᵐ is one cycle of length 10ᵐ; its first return to the multiples of 10 is the odometer one scale down (the adding machine on the 10-adics, which is self-similar). On ℤ/2ⁿ, every `x → a·x + b` with a odd has cycle lengths that are powers of two.

In [4]:
for m in (2, 3, 4):
    print(m, M.odometer_first_return(m)['is_odometer_one_scale_down'], M.affine_cycle_type(1, 1, 10**m))
print({(a, b): M.solenoid_cycles(a, b, 8) for a in (1, 3, 5, 7) for b in (0, 1)})

2 True {100: 1}
3 True {1000: 1}
4 True {10000: 1}
{(1, 0): {1: 256}, (1, 1): {256: 1}, (3, 0): {1: 2, 64: 2, 32: 2, 16: 2, 8: 2, 4: 2, 2: 3}, (3, 1): {128: 2}, (5, 0): {1: 4, 64: 2, 32: 2, 16: 2, 8: 2, 4: 2, 2: 2}, (5, 1): {256: 1}, (7, 0): {1: 2, 32: 4, 16: 4, 8: 4, 4: 4, 2: 7}, (7, 1): {64: 4}}


**Open item shown honestly.** The columns are rotors only for the *wiring* `num1 × digit`; that wiring is a bijection of ℤ/10ᵐ only when `num1` is coprime to 10. For other `num1` the column map is many-to-one, and `cycle_type` raises rather than report a cycle type that does not exist:

In [5]:
try:
    M.affine_cycle_type(4, 1, 100)
except ValueError as e:
    print('ValueError:', e)

ValueError: not a permutation
